# 🥥 Coconut Leaf Color Detection - v1

## Model: `coconut_leaf_color_v1`

### Goal
Train a binary classifier that identifies coconut leaf color — **green** (healthy) vs **yellow** (yellowing/stressed).

### Key Design Decisions
- **Architecture:** MobileNetV2 (Transfer Learning) — lightweight, strong color feature extraction
- **Classes:** `green` (healthy-leaves) | `yellow` (unhealthy-yellowing)
- **Color-Preserving Augmentation:** Geometric-only — NO brightness/hue shifts (preserves green vs yellow signal)
- **Anti-Overfitting:** Dropout + BatchNorm + L2 Regularization + EarlyStopping + ReduceLROnPlateau + Label Smoothing
- **Focal Loss:** Handles class imbalance (green: 4,572 | yellow: 4,028)
- **Two-Phase Training:** Frozen base → Fine-tuning last layers

### Data
- `healthy-leaves/`       → `green` class
- `unhealthy-yellowing/`  → `yellow` class

### Target
- **Accuracy:** ≥90%
- **Macro F1:** ≥88%
- **Train-Val Gap:** <7% (no overfitting)

## 1. Setup & Imports

In [ ]:
import os
import shutil
import random
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 2. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.abspath('..')
DATA_DIR        = os.path.join(BASE_DIR, 'data', 'raw')
GREEN_SRC       = os.path.join(DATA_DIR, 'healthy-leaves')       # green class
YELLOW_SRC      = os.path.join(DATA_DIR, 'unhealthy-yellowing')  # yellow class
DATASET_DIR     = os.path.join(BASE_DIR, 'data', 'processed', 'leaf_color_v1')
MODEL_SAVE_DIR  = os.path.join(BASE_DIR, 'models', 'coconut_leaf_color_v1')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE        = 224
BATCH_SIZE      = 32
PHASE1_EPOCHS   = 20       # Frozen base — learn classification head
PHASE2_EPOCHS   = 20       # Fine-tune last 30 layers
LR_PHASE1       = 1e-3
LR_PHASE2       = 5e-5     # Low LR — prevent catastrophic forgetting
DROPOUT_RATE    = 0.5      # Higher dropout for stronger regularization
L2_REG          = 2e-4     # L2 weight decay
LABEL_SMOOTHING = 0.1      # Prevents overconfident predictions
FINE_TUNE_AT    = 100      # MobileNetV2: unfreeze layers after this index

# ── Classes ───────────────────────────────────────────────────────────────
# green = healthy leaves (green color)
# yellow = yellowing/stressed leaves
CLASSES         = ['green', 'yellow']   # alphabetical → ImageDataGenerator order
NUM_CLASSES     = 2

# ── Model save paths ──────────────────────────────────────────────────────
BEST_MODEL_PATH   = os.path.join(MODEL_SAVE_DIR, 'best_model.keras')
PHASE1_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'phase1_best.keras')

print(f'Dataset dir : {DATASET_DIR}')
print(f'Model dir   : {MODEL_SAVE_DIR}')
print(f'Classes     : {CLASSES}')

## 3. Build Dataset Directory Structure

Copies `healthy-leaves` → `green/` and `unhealthy-yellowing` → `yellow/`

In [ ]:
def copy_split(src_class_dir, split_name, dest_class_name, dataset_dir):
    """
    src_class_dir  : source root (e.g. healthy-leaves/)
    split_name     : 'training' | 'validation' | 'test'
    dest_class_name: 'green' | 'yellow'
    """
    split_map = {'training': 'train', 'validation': 'val', 'test': 'test'}
    dest_split = split_map.get(split_name, split_name)

    src  = os.path.join(src_class_dir, split_name)
    dest = os.path.join(dataset_dir, dest_split, dest_class_name)
    os.makedirs(dest, exist_ok=True)

    if not os.path.exists(src):
        print(f'  [SKIP] {src} not found')
        return 0

    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    imgs = [f for f in os.listdir(src) if os.path.splitext(f)[1].lower() in exts]
    copied = 0
    for f in imgs:
        dst_path = os.path.join(dest, f)
        if not os.path.exists(dst_path):
            shutil.copy2(os.path.join(src, f), dst_path)
        copied += 1
    return copied


if os.path.exists(DATASET_DIR):
    print('Dataset already exists — skipping copy.')
else:
    print('Building dataset structure...')
    for split in ['training', 'validation', 'test']:
        n_g = copy_split(GREEN_SRC,  split, 'green',  DATASET_DIR)
        n_y = copy_split(YELLOW_SRC, split, 'yellow', DATASET_DIR)
        print(f'  {split:12s}  green={n_g}  yellow={n_y}')
    print('Done.')

# Count images
print()
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        p = os.path.join(DATASET_DIR, split, cls)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f'{split:5s}/{cls:7s}: {n} images')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
counts = {}
for split in ['train', 'val', 'test']:
    counts[split] = {}
    for cls in CLASSES:
        p = os.path.join(DATASET_DIR, split, cls)
        counts[split][cls] = len(os.listdir(p)) if os.path.exists(p) else 0

print(f"{'Split':8s} {'green':>8s} {'yellow':>8s} {'total':>8s}")
print('-' * 38)
for split, cls_dict in counts.items():
    total = sum(cls_dict.values())
    print(f"{split:8s} {cls_dict['green']:>8d} {cls_dict['yellow']:>8d} {total:>8d}")

ratio = counts['train']['green'] / max(counts['train']['yellow'], 1)
print(f'\nTrain green/yellow ratio: {ratio:.2f}')

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#27ae60', '#f1c40f']
for ax, split in zip(axes, ['train', 'val', 'test']):
    vals = [counts[split][c] for c in CLASSES]
    bars = ax.bar(CLASSES, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{split.upper()} split', fontsize=13, fontweight='bold')
    ax.set_ylabel('Images')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Dataset Distribution — coconut_leaf_color_v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. HSV Color Analysis

Verifies the green/yellow color signal exists in the data — validates our task is feasible.

In [ ]:
def hsv_color_analysis(img_path):
    """Returns (green_pct, yellow_pct, brown_pct) for an image."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.resize(img, (224, 224))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    total = hsv.shape[0] * hsv.shape[1]

    # Green: H=35–90
    mask_g = cv2.inRange(hsv, np.array([35, 40, 40]), np.array([90, 255, 255]))
    # Yellow: H=15–35
    mask_y = cv2.inRange(hsv, np.array([15, 40, 40]), np.array([35, 255, 255]))
    # Brown/orange: H=0–15
    mask_b = cv2.inRange(hsv, np.array([0, 30, 30]),  np.array([15, 255, 200]))

    return (
        np.sum(mask_g > 0) / total * 100,
        np.sum(mask_y > 0) / total * 100,
        np.sum(mask_b > 0) / total * 100,
    )


def sample_color(class_dir, n=200):
    files = [os.path.join(class_dir, f)
             for f in os.listdir(class_dir)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(files)
    results = [hsv_color_analysis(f) for f in files[:n]]
    results = [r for r in results if r]
    return (
        [r[0] for r in results],
        [r[1] for r in results],
        [r[2] for r in results],
    )


print('Analyzing color channels on training samples...')
g_green, g_yellow, g_brown = sample_color(os.path.join(DATASET_DIR, 'train', 'green'))
y_green, y_yellow, y_brown = sample_color(os.path.join(DATASET_DIR, 'train', 'yellow'))

print(f'\nGreen class  — green%: {np.mean(g_green):.1f}  yellow%: {np.mean(g_yellow):.1f}  brown%: {np.mean(g_brown):.1f}')
print(f'Yellow class — green%: {np.mean(y_green):.1f}  yellow%: {np.mean(y_yellow):.1f}  brown%: {np.mean(y_brown):.1f}')

# Grouped bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names   = ['Green Channel %', 'Yellow Channel %', 'Brown Channel %']
green_cls_vals  = [np.mean(g_green), np.mean(g_yellow), np.mean(g_brown)]
yellow_cls_vals = [np.mean(y_green), np.mean(y_yellow), np.mean(y_brown)]
bar_green  = ['#27ae60', '#f1c40f', '#8B4513']
bar_yellow = ['#2ecc71', '#e67e22', '#a0522d']

for ax, name, gv, yv, gc, yc in zip(
    axes, channel_names,
    green_cls_vals, yellow_cls_vals,
    bar_green, bar_yellow
):
    bars = ax.bar(['green class', 'yellow class'], [gv, yv], color=[gc, yc],
                  edgecolor='white', linewidth=1.5)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Average %')
    for bar, val in zip(bars, [gv, yv]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('HSV Color Channel Analysis — green vs yellow class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'color_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Sample Image Visualization

In [ ]:
def show_samples(class_dir, label, border_color, axes_row, n=8):
    files = [
        os.path.join(class_dir, f) for f in os.listdir(class_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    random.shuffle(files)
    files = files[:n]
    for j, ax in enumerate(axes_row):
        if j >= len(files):
            ax.axis('off')
            continue
        result = hsv_color_analysis(files[j])
        g_pct  = f'{result[0]:.0f}% G' if result else ''
        y_pct  = f'{result[1]:.0f}% Y' if result else ''
        img    = Image.open(files[j]).convert('RGB').resize((224, 224))
        ax.imshow(img)
        ax.set_title(f'{label}\n{g_pct} | {y_pct}', color=border_color, fontsize=7, fontweight='bold')
        ax.axis('off')


N = 8
fig, axes = plt.subplots(2, N, figsize=(20, 5))
show_samples(os.path.join(DATASET_DIR, 'train', 'green'),  'GREEN',  '#27ae60', axes[0], N)
show_samples(os.path.join(DATASET_DIR, 'train', 'yellow'), 'YELLOW', '#f39c12', axes[1], N)
plt.suptitle('Sample Training Images (G% = green pixels | Y% = yellow pixels)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'sample_images.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Data Generators (Color-Preserving Augmentation)

> **Critical:** No brightness/hue/channel-shift augmentation — the green vs yellow color difference IS the classification signal.

In [ ]:
# ── Train: geometric-only augmentation ─────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=25,
    width_shift_range=0.12,
    height_shift_range=0.12,
    horizontal_flip=True,
    vertical_flip=False,
    zoom_range=0.15,
    shear_range=0.08,
    fill_mode='reflect',
    # NO brightness_range / channel_shift — must preserve color signal!
)

# ── Val / Test: only rescale ───────────────────────────────────────────────
eval_datagen = ImageDataGenerator(rescale=1.0 / 255)


def make_gen(datagen, split, shuffle=True):
    return datagen.flow_from_directory(
        os.path.join(DATASET_DIR, split),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=shuffle,
        seed=SEED,
    )


train_gen = make_gen(train_datagen, 'train', shuffle=True)
val_gen   = make_gen(eval_datagen,  'val',   shuffle=False)
test_gen  = make_gen(eval_datagen,  'test',  shuffle=False)

print('Class indices:', train_gen.class_indices)
print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)} | Test batches: {len(test_gen)}')

# Class weights for imbalance
n_green  = np.sum(train_gen.classes == train_gen.class_indices['green'])
n_yellow = np.sum(train_gen.classes == train_gen.class_indices['yellow'])
total    = n_green + n_yellow

class_weight = {
    train_gen.class_indices['green']:  total / (2 * n_green),
    train_gen.class_indices['yellow']: total / (2 * n_yellow),
}
print(f'\nClass weights: {class_weight}')
print(f'  green: {n_green}  yellow: {n_yellow}')

## 8. Focal Loss + Label Smoothing

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    """
    Focal loss with label smoothing.
    Down-weights easy examples and focuses training on hard ones.
    """
    def focal_loss_fn(y_true, y_pred):
        # Label smoothing — prevents the model from being overconfident
        y_true_smooth = y_true * (1 - LABEL_SMOOTHING) + (LABEL_SMOOTHING / NUM_CLASSES)
        epsilon = tf.keras.backend.epsilon()
        y_pred  = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        cross_entropy = -y_true_smooth * tf.math.log(y_pred)
        focal_weight  = alpha * y_true_smooth * tf.math.pow(1.0 - y_pred, gamma)
        loss          = focal_weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=1))

    return focal_loss_fn


print('Focal loss defined (gamma=2.0, alpha=0.25, label_smoothing=0.1)')

## 9. Model Architecture (MobileNetV2 + Anti-Overfitting Head)

MobileNetV2 pre-trained on ImageNet. Two-phase training:
- **Phase 1:** Frozen base — only the classification head is trained
- **Phase 2:** Fine-tune last layers with very low LR

In [ ]:
def build_model(trainable_base=False, fine_tune_at=None):
    """
    MobileNetV2 base + custom classification head.
    Anti-overfitting: Dropout + BatchNorm + L2 regularization.
    """
    base = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )

    if not trainable_base:
        base.trainable = False
    elif fine_tune_at is not None:
        base.trainable = True
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False
    else:
        base.trainable = True

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = base(inp, training=False)

    # Pooling + Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)                                  # 0.5 dropout

    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        kernel_initializer='he_normal',
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE * 0.6)(x)                           # 0.3 dropout

    x = layers.Dense(
        128, activation='relu',
        kernel_regularizer=regularizers.l2(L2_REG),
        kernel_initializer='he_normal',
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return keras.Model(inp, out), base


model, base_model = build_model(trainable_base=False)

trainable_count = sum(tf.size(w).numpy() for w in model.trainable_weights)
total_count     = sum(tf.size(w).numpy() for w in model.weights)
print(f'Total params     : {total_count:,}')
print(f'Trainable params : {trainable_count:,}  ({100*trainable_count/total_count:.1f}%)')
model.summary()

## 10. Phase 1 — Train Classification Head (Frozen Base)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE1),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy'],
)

callbacks_phase1 = [
    ModelCheckpoint(
        PHASE1_MODEL_PATH,
        monitor='val_accuracy', save_best_only=True, verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=6,
        restore_best_weights=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.4, patience=3,
        min_lr=1e-6, verbose=1,
    ),
]

print(f'Phase 1: Training head (frozen base) — max {PHASE1_EPOCHS} epochs, LR={LR_PHASE1}')
history1 = model.fit(
    train_gen,
    epochs=PHASE1_EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    class_weight=class_weight,
    verbose=1,
)

## 11. Phase 2 — Fine-tuning (Unfreeze Last Layers)

In [ ]:
# Unfreeze layers after FINE_TUNE_AT index
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

n_tunable = sum(1 for l in base_model.layers if l.trainable)
total_base = len(base_model.layers)
print(f'MobileNetV2 total layers : {total_base}')
print(f'Fine-tuning              : {n_tunable} layers (after index {FINE_TUNE_AT})')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE2),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy'],
)

callbacks_phase2 = [
    ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor='val_accuracy', save_best_only=True, verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=4,
        min_lr=1e-7, verbose=1,
    ),
]

print(f'Phase 2: Fine-tuning — max {PHASE2_EPOCHS} epochs, LR={LR_PHASE2}')
history2 = model.fit(
    train_gen,
    epochs=PHASE2_EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    class_weight=class_weight,
    verbose=1,
)

## 12. Training History Plots

In [ ]:
def merge_history(h1, h2):
    return {k: h1.history[k] + h2.history[k] for k in h1.history}


combined = merge_history(history1, history2)
p1_len   = len(history1.history['accuracy'])
epochs   = range(1, len(combined['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax = axes[0]
ax.plot(epochs, combined['accuracy'],     color='#27ae60', linewidth=2, label='Train Acc')
ax.plot(epochs, combined['val_accuracy'], color='#27ae60', linewidth=2,
        linestyle='--', label='Val Acc')
ax.axvline(p1_len, color='gray', linestyle=':', linewidth=1.5,
           label=f'Phase 2 start (ep {p1_len})')
ax.set_title('Accuracy', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Loss
ax = axes[1]
ax.plot(epochs, combined['loss'],     color='#e74c3c', linewidth=2, label='Train Loss')
ax.plot(epochs, combined['val_loss'], color='#e74c3c', linewidth=2,
        linestyle='--', label='Val Loss')
ax.axvline(p1_len, color='gray', linestyle=':', linewidth=1.5,
           label=f'Phase 2 start (ep {p1_len})')
ax.set_title('Loss', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle('Training History — coconut_leaf_color_v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

# Overfitting check
final_train = combined['accuracy'][-1]
final_val   = combined['val_accuracy'][-1]
gap         = final_train - final_val
print(f'Final train accuracy : {final_train:.4f}')
print(f'Final val accuracy   : {final_val:.4f}')
print(f'Train-val gap        : {gap:.4f}  ({"✅ Good generalization" if gap < 0.07 else "⚠️ Possible overfitting"})')

## 13. Load Best Model & Evaluate on Test Set

In [ ]:
print(f'Loading best model: {BEST_MODEL_PATH}')
best_model = keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={'focal_loss_fn': focal_loss(gamma=2.0, alpha=0.25)},
)

test_gen_eval = eval_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

test_loss, test_accuracy = best_model.evaluate(test_gen_eval, verbose=1)
print(f'\nTest Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_accuracy:.4f}  ({test_accuracy*100:.2f}%)')

## 14. Per-Class Metrics

In [ ]:
test_gen_eval.reset()
y_pred_proba = best_model.predict(test_gen_eval, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = test_gen_eval.classes

class_idx = test_gen_eval.class_indices
print('Class indices:', class_idx)

precision  = precision_score(y_true, y_pred, average=None)
recall     = recall_score(y_true,    y_pred, average=None)
f1         = f1_score(y_true,        y_pred, average=None)
macro_p    = precision_score(y_true, y_pred, average='macro')
macro_r    = recall_score(y_true,    y_pred, average='macro')
macro_f1   = f1_score(y_true,        y_pred, average='macro')

print(f"\n{'Class':8s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s}")
print('-' * 40)
for i, cls in enumerate(CLASSES):
    print(f"{cls:8s} {precision[i]:>10.4f} {recall[i]:>8.4f} {f1[i]:>8.4f}")
print('-' * 40)
print(f"{'Macro':8s} {macro_p:>10.4f} {macro_r:>8.4f} {macro_f1:>8.4f}")
print(f"\nTest Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x     = np.arange(len(CLASSES))
width = 0.25
ax.bar(x - width, precision, width, label='Precision', color='#3498db')
ax.bar(x,         recall,    width, label='Recall',    color='#27ae60')
ax.bar(x + width, f1,        width, label='F1',        color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Per-Class Metrics — Test Set', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
    ax.text(i - width, p + 0.01, f'{p:.3f}', ha='center', fontsize=8)
    ax.text(i,         r + 0.01, f'{r:.3f}', ha='center', fontsize=8)
    ax.text(i + width, f + 0.01, f'{f:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'per_class_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 15. Confusion Matrix

In [ ]:
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, fmt, title in zip(
    axes, [cm, cm_pct], ['d', '.1f'],
    ['Confusion Matrix (Counts)', 'Confusion Matrix (%)'],
):
    sns.heatmap(
        data, annot=True, fmt=fmt, ax=ax,
        xticklabels=CLASSES, yticklabels=CLASSES,
        cmap='RdYlGn', linewidths=0.5, linecolor='white',
        annot_kws={'size': 14, 'weight': 'bold'},
    )
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual',    fontsize=12)
    ax.set_title(title, fontweight='bold')

plt.suptitle('Confusion Matrix — coconut_leaf_color_v1 (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total   = cm[i].sum()
    print(f'{cls:8s}: {correct}/{total}  ({100*correct/total:.1f}% recall)')

## 16. Classification Report

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
print('=' * 60)
print('CLASSIFICATION REPORT — coconut_leaf_color_v1')
print('=' * 60)
print(report)

with open(os.path.join(MODEL_SAVE_DIR, 'classification_report.txt'), 'w') as f:
    f.write('CLASSIFICATION REPORT — coconut_leaf_color_v1\n')
    f.write('=' * 60 + '\n')
    f.write(report)

## 17. Sample Predictions Visualization

In [ ]:
def visualize_predictions(model, gen, n_correct=6, n_wrong=6):
    gen.reset()
    all_imgs, all_true, all_pred, all_conf = [], [], [], []

    for imgs, labels in gen:
        preds = model.predict(imgs, verbose=0)
        for i in range(len(imgs)):
            all_imgs.append(imgs[i])
            all_true.append(np.argmax(labels[i]))
            all_pred.append(np.argmax(preds[i]))
            all_conf.append(np.max(preds[i]))
        if len(all_imgs) >= 300:
            break

    correct_idx = [i for i in range(len(all_true)) if all_true[i] == all_pred[i]]
    wrong_idx   = [i for i in range(len(all_true)) if all_true[i] != all_pred[i]]
    random.shuffle(correct_idx)
    random.shuffle(wrong_idx)

    total_cols = max(n_correct, n_wrong)
    fig, axes  = plt.subplots(2, total_cols, figsize=(total_cols * 2.5, 6))

    for row, (lbl, indices, color) in enumerate([
        ('Correct', correct_idx[:n_correct], '#27ae60'),
        ('Wrong',   wrong_idx[:n_wrong],     '#e74c3c'),
    ]):
        for col in range(total_cols):
            ax = axes[row][col]
            if col >= len(indices):
                ax.axis('off')
                continue
            idx = indices[col]
            ax.imshow(all_imgs[idx])
            ax.set_title(
                f'True: {CLASSES[all_true[idx]]}\nPred: {CLASSES[all_pred[idx]]}\n{all_conf[idx]*100:.1f}%',
                color=color, fontsize=7, fontweight='bold',
            )
            ax.axis('off')
        axes[row][0].set_ylabel(lbl, fontsize=10, fontweight='bold', color=color)

    plt.suptitle('Sample Predictions — coconut_leaf_color_v1', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_DIR, 'sample_predictions.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Correct samples: {len(correct_idx[:n_correct])} shown')
    print(f'Wrong samples  : {len(wrong_idx[:n_wrong])} shown')


visualize_predictions(best_model, test_gen_eval)

## 18. Confidence Score Distribution

In [ ]:
test_gen_eval.reset()
all_proba  = best_model.predict(test_gen_eval, verbose=1)
confidence = np.max(all_proba, axis=1)
all_pred2  = np.argmax(all_proba, axis=1)
all_true2  = test_gen_eval.classes

correct_conf   = confidence[all_pred2 == all_true2]
incorrect_conf = confidence[all_pred2 != all_true2]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(correct_conf,   bins=30, color='#27ae60', alpha=0.7,
        label=f'Correct ({len(correct_conf)})', edgecolor='white')
ax.hist(incorrect_conf, bins=30, color='#e74c3c', alpha=0.7,
        label=f'Incorrect ({len(incorrect_conf)})', edgecolor='white')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Prediction Confidence Distribution', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'confidence_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean confidence (correct)  : {correct_conf.mean():.4f}')
print(f'Mean confidence (incorrect): {incorrect_conf.mean():.4f}')

## 19. Save model_info.json

In [ ]:
import datetime

train_total = sum(counts['train'].values())
val_total   = sum(counts['val'].values())
test_total  = sum(counts['test'].values())

model_info = {
    "model_name":   "coconut_leaf_color_v1",
    "version":      "v1",
    "created_at":   datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    "architecture": "MobileNetV2",
    "task":         "Coconut leaf color detection (green vs yellow, binary)",
    "classes":      CLASSES,
    "num_classes":  NUM_CLASSES,
    "input_shape":  [IMG_SIZE, IMG_SIZE, 3],

    "training": {
        "phase1_epochs":        len(history1.history['accuracy']),
        "phase2_epochs":        len(history2.history['accuracy']),
        "batch_size":           BATCH_SIZE,
        "lr_phase1":            LR_PHASE1,
        "lr_phase2":            LR_PHASE2,
        "dropout_rate":         DROPOUT_RATE,
        "l2_regularization":    L2_REG,
        "label_smoothing":      LABEL_SMOOTHING,
        "fine_tune_at_layer":   FINE_TUNE_AT,
        "loss_function":        "Focal Loss (gamma=2.0, alpha=0.25) + Label Smoothing",
        "optimizer":            "Adam",
        "augmentation":         "Geometric only (NO color aug — preserves green/yellow signal)",
        "anti_overfitting":     [
            "Dropout (0.5 + 0.3 + 0.2)",
            "BatchNormalization",
            "L2 regularization (2e-4)",
            "EarlyStopping",
            "ReduceLROnPlateau",
            "Label Smoothing (0.1)",
            "Color-preserving augmentation",
            "Class weights",
        ],
        "final_train_acc":      round(float(combined['accuracy'][-1]), 4),
        "final_val_acc":        round(float(combined['val_accuracy'][-1]), 4),
        "train_val_gap":        round(float(combined['accuracy'][-1] - combined['val_accuracy'][-1]), 4),
    },

    "data": {
        "train_samples":  train_total,
        "val_samples":    val_total,
        "test_samples":   test_total,
        "train_green":    counts['train']['green'],
        "train_yellow":   counts['train']['yellow'],
        "sources": [
            "healthy-leaves/  → green class",
            "unhealthy-yellowing/  → yellow class",
        ],
    },

    "test_performance": {
        "accuracy":          round(float(test_accuracy), 4),
        "macro_precision":   round(float(macro_p),  4),
        "macro_recall":      round(float(macro_r),  4),
        "macro_f1":          round(float(macro_f1), 4),
        "green_precision":   round(float(precision[class_idx['green']]),  4),
        "green_recall":      round(float(recall[class_idx['green']]),     4),
        "green_f1":          round(float(f1[class_idx['green']]),         4),
        "yellow_precision":  round(float(precision[class_idx['yellow']]), 4),
        "yellow_recall":     round(float(recall[class_idx['yellow']]),    4),
        "yellow_f1":         round(float(f1[class_idx['yellow']]),        4),
    },

    "files": {
        "best_model":   "best_model.keras",
        "phase1_model": "phase1_best.keras",
        "model_info":   "model_info.json",
    },
    "api_endpoint": "/predict/leaf-color",
}

model_info_path = os.path.join(MODEL_SAVE_DIR, 'model_info.json')
with open(model_info_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f'model_info.json saved: {model_info_path}')
print(json.dumps(model_info, indent=2))

## 20. Final Summary

In [ ]:
gap = combined['accuracy'][-1] - combined['val_accuracy'][-1]

print('=' * 60)
print('      coconut_leaf_color_v1 — TRAINING COMPLETE')
print('=' * 60)
print(f'Architecture  : MobileNetV2 (Transfer Learning)')
print(f'Task          : Leaf color detection (green vs yellow)')
print(f'Input size    : {IMG_SIZE}x{IMG_SIZE}x3')
print(f'Classes       : {CLASSES}')
print()
print(f'Training data : {train_total:,} images')
print(f'  green       : {counts["train"]["green"]:,}')
print(f'  yellow      : {counts["train"]["yellow"]:,}')
print()
print(f'Test Accuracy : {test_accuracy*100:.2f}%')
print(f'Macro F1      : {macro_f1*100:.2f}%')
print(f'Macro Recall  : {macro_r*100:.2f}%')
print(f'Macro Prec.   : {macro_p*100:.2f}%')
print()
print(f'Train acc     : {combined["accuracy"][-1]*100:.2f}%')
print(f'Val acc       : {combined["val_accuracy"][-1]*100:.2f}%')
print(f'Train-val gap : {gap:.4f}  ({"✅ Good" if gap < 0.07 else "⚠️ Overfit risk"})')
print()
print(f'Anti-overfitting measures applied:')
print(f'  - Dropout: 0.5 + 0.3 + 0.2 (3 stages)')
print(f'  - BatchNormalization after each dense layer')
print(f'  - L2 regularization: {L2_REG}')
print(f'  - Label smoothing: {LABEL_SMOOTHING}')
print(f'  - EarlyStopping + ReduceLROnPlateau')
print(f'  - Color-preserving augmentation (no hue/brightness shift)')
print()
print(f'Model saved   : {MODEL_SAVE_DIR}')
print('Files:')
for fname in os.listdir(MODEL_SAVE_DIR):
    size_mb = os.path.getsize(os.path.join(MODEL_SAVE_DIR, fname)) / (1024 * 1024)
    print(f'  {fname:<42s} {size_mb:.1f} MB')
print('=' * 60)